In [ ]:
# jhofydu/kafka-cpu-sliding-producer:v0.1.0
import os
import time
import json
import datetime
from collections import deque
from kafka import KafkaProducer
from prometheus_api_client import PrometheusConnect

# Configuration (edit env vars for different settings)
PROM_URL = os.getenv(
    "PROM_URL", "http://prometheus-operated.prometheus.svc.cluster.local:9090"
)
KAFKA_BOOTSTRAP_SERVERS = os.getenv(
    "KAFKA_BOOTSTRAP_SERVERS", "kafka.apache-kafka.svc.cluster.local:9092"
)
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC", "sliding_window_lstm")
SCRAPE_INTERVAL_SEC = int(os.getenv("SCRAPE_INTERVAL_SEC", "5"))
SLIDING_WINDOW_SIZE = int(os.getenv("SLIDING_WINDOW_SIZE", "5"))

# This query is tailored to your node's instance label
PROM_QUERY = '100 * (1 - avg(rate(node_cpu_seconds_total{mode="idle",instance="172.19.0.3:9100"}[1m])) by (instance))'


def get_latest_cpu_point(prom, query, interval_sec, last_seen_ts):
    """
    Query Prometheus for the current CPU percentage for your node.
    Only accepts a value if its timestamp is aligned and not duplicated.
    """
    try:
        results = prom.custom_query(query=query)
    except Exception as e:
        print(f"[ERROR] Prometheus query failed: {e}")
        return None, None

    if not results:
        return None, None

    for sample in results:
        cpu_val = float(sample["value"][1])
        prom_ts = int(float(sample["value"][0]))

        # Only accept timestamps aligned with interval (e.g., :00, :05, ...)
        if prom_ts % interval_sec != 0:
            continue

        # Skip duplicate timestamps
        if last_seen_ts is not None and prom_ts == last_seen_ts:
            continue

        return prom_ts, cpu_val

    return None, None


def main():
    prom = PrometheusConnect(url=PROM_URL, disable_ssl=True)
    producer = KafkaProducer(
        bootstrap_servers=[
            s.strip() for s in KAFKA_BOOTSTRAP_SERVERS.split(",") if s.strip()
        ],
        value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    )

    sliding_window = deque(maxlen=SLIDING_WINDOW_SIZE)
    last_seen_ts = None

    print("============ CONFIG ============")
    print(f"Prometheus: {PROM_URL}")
    print(f"Query:      {PROM_QUERY}")
    print(f"Kafka:      {KAFKA_BOOTSTRAP_SERVERS}")
    print(f"Topic:      {KAFKA_TOPIC}")
    print(f"Interval:   {SCRAPE_INTERVAL_SEC} sec")
    print(f"Window size:{SLIDING_WINDOW_SIZE}")
    print("================================\n")

    while True:
        prom_ts, cpu_pct = get_latest_cpu_point(
            prom, PROM_QUERY, SCRAPE_INTERVAL_SEC, last_seen_ts
        )
        if prom_ts is None:
            time.sleep(2)
            continue

        ts_human = datetime.datetime.fromtimestamp(prom_ts).strftime(
            "%Y-%m-%d %H:%M:%S"
        )
        print(f"[WINDOW] ADD: {ts_human}, {cpu_pct:.4f}")

        # Add data point to the sliding window
        sliding_window.append({"ts": ts_human, "cpu_pct": cpu_pct})

        # Only publish when window is full
        if len(sliding_window) == SLIDING_WINDOW_SIZE:
            window_data = list(sliding_window)
            # Minimal human-readable print, just the window contents:
            print(
                " | ".join(
                    f"{d['ts'].split()[-1]} {d['cpu_pct']:.2f}" for d in window_data
                )
            )
            try:
                producer.send(KAFKA_TOPIC, window_data)
                producer.flush()
            except Exception as ex:
                print(f"[ERROR] Failed to publish to Kafka: {ex}")

        last_seen_ts = prom_ts
        time.sleep(SCRAPE_INTERVAL_SEC)


if __name__ == "__main__":
    print("Sliding window producer started.")
    main()


Sliding window producer started.
============ CONFIG ============
Prometheus: http://prometheus-operated.prometheus.svc.cluster.local:9090
Query:      100 * (1 - avg(rate(node_cpu_seconds_total{mode="idle",instance="172.19.0.3:9100"}[1m])) by (instance))
Kafka:      kafka.apache-kafka.svc.cluster.local:9092
Topic:      sliding_window
Interval:   5 sec
Window size:5

[WINDOW] ADD: 2025-09-24 14:35:25, 20.8687
[WINDOW] ADD: 2025-09-24 14:35:30, 20.8687
[WINDOW] ADD: 2025-09-24 14:35:35, 20.8687
[WINDOW] ADD: 2025-09-24 14:35:40, 20.8687
[WINDOW] ADD: 2025-09-24 14:35:45, 20.8687
14:35:25 20.87 | 14:35:30 20.87 | 14:35:35 20.87 | 14:35:40 20.87 | 14:35:45 20.87
[WINDOW] ADD: 2025-09-24 14:35:50, 20.4422
14:35:30 20.87 | 14:35:35 20.87 | 14:35:40 20.87 | 14:35:45 20.87 | 14:35:50 20.44
[WINDOW] ADD: 2025-09-24 14:35:55, 20.4422
14:35:35 20.87 | 14:35:40 20.87 | 14:35:45 20.87 | 14:35:50 20.44 | 14:35:55 20.44
[WINDOW] ADD: 2025-09-24 14:36:00, 20.4422
14:35:40 20.87 | 14:35:45 20.87 | 14:3